# ArchiGen 3D — Run 3 on Kaggle (recipe v3 + diffusion, ~10.5 h)

**Goal:** beat the original ArchiGen models on every measured axis, with one protocol for all models.

| GPU | Jobs (in order) | Cap |
|---|---|---|
| 0 | **Pix2Pix v3**: 2-scale discriminator (pix2pixHD), checkpoint chosen on **val KID** → **facade parser** (DeepLabV3, for the FCN-score) | 5.5 h + 40 min |
| 1 | **UNet++ floor plans** fine-tuned from Run 2: **Lovász-softmax** on the room head (optimises room mIoU), **×3 oversampling of plans with rare rooms** (dining, office, technical), **drawing-style augmentation** → **ControlNet** (Stable Diffusion 1.5) fine-tuned on the CMP label maps | 4 h + 4.5 h |

**Evaluation (test split, once, same protocol for every facade model):** KID, LPIPS, L1/PSNR/SSIM **and FCN-score**
(layout fidelity: a parser trained on real photos checks that the generated facade really has the requested windows,
pillars, balconies…). Facade models scored: Run 2 Pix2Pix, Run 2 L1 baseline, Run 3 Pix2Pix, **untrained ControlNet
(what the original project showed)**, **fine-tuned ControlNet**, and the real photos (upper bound).
Floor plans: single pass, 4-fold TTA, **restyled test** (Run 2 and Run 3), door/window F1 + 3D shell, external plans.

**Before running:** *Session options* → **Accelerator: GPU T4 ×2**, **Internet: On**. **+ Add Input**:
1. the code dataset — rebuild it with `python kaggle/make_bundle.py --part facades` and upload it as a **new version**;
2. the public **CubiCasa5k** dataset;
3. your **Run 2 notebook output** (*+ Add Input → Your Work → Notebooks*): the floor-plan model starts from it and the Run 2 facade models are re-scored with the FCN-score.

Then **Save Version → Save & Run All** (~10.5 h of your 30 h weekly GPU quota).

In [ ]:
# 1) Setup: inputs, writable workspace, Run 2 checkpoints, libraries, GPUs
import glob, os, shutil, subprocess, sys, time, json, zipfile
from pathlib import Path

def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return Path(hits[0]) if hits else None

code_root = find('/kaggle/input/**/archigen/params.yaml')
FACADES = find('/kaggle/input/**/archigen/data/raw/facades')
CUBICASA = find('/kaggle/input/**/archigen/data/processed/cubicasa_cache')
assert code_root, "Code not found: '+ Add Input' -> the dataset made from archigen_facades.zip"
SRC = code_root.parent
assert (SRC / 'src/training/train_controlnet.py').exists(), \
    'This code bundle predates the new Run 3: rebuild it with `python kaggle/make_bundle.py --part facades` and re-upload.'

W = Path('/kaggle/working/archigen')
if W.exists():
    shutil.rmtree(W)
(W / 'data/raw').mkdir(parents=True)
(W / 'data/processed').mkdir(parents=True)
(W / 'models/checkpoints').mkdir(parents=True)
shutil.copytree(SRC / 'src', W / 'src')
shutil.copy(SRC / 'params.yaml', W / 'params.yaml')
for name in ('facades', 'dev_external_plans', 'real_external_plans', 'external_test_plans'):
    if (SRC / 'data/raw' / name).exists():
        os.symlink(SRC / 'data/raw' / name, W / 'data/raw' / name)
if CUBICASA:
    os.symlink(CUBICASA, W / 'data/processed/cubicasa_cache')
else:
    raw = [Path(t).parent for t in glob.glob('/kaggle/input/**/train.txt', recursive=True)
           if (Path(t).parent / 'val.txt').exists() and (Path(t).parent / 'test.txt').exists()]
    if raw:
        print('Rasterizing CubiCasa5K from', raw[0], '(official splits)...')
        r = subprocess.run([sys.executable, '-m', 'src.data.rasterize_cubicasa', '--source', str(raw[0]),
                            '--out', str(W / 'data/processed/cubicasa_cache'), '--workers', str(os.cpu_count() or 4)],
                           cwd=W, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
HAS_FACADES = FACADES is not None
HAS_FP = any((W / 'data/processed/cubicasa_cache/train').glob('*.npz')) if (W / 'data/processed/cubicasa_cache/train').exists() else False

# Run 2 checkpoints (files, or inside the Run 2 output zip) -> *_run2.pt
CK = W / 'models/checkpoints'
RUN2 = {'floorplan_unetpp_best.pt': 'floorplan_unetpp_run2.pt', 'facade_pix2pix_best.pt': 'facade_pix2pix_run2.pt',
        'facade_unet_l1_best.pt': 'facade_unet_l1_run2.pt'}
zips = sorted(glob.glob('/kaggle/input/**/archigen_results_run2.zip', recursive=True))
for src_name, dst_name in RUN2.items():
    hit = find(f'/kaggle/input/**/{src_name}')
    if hit:
        shutil.copy(hit, CK / dst_name)
    elif zips:
        with zipfile.ZipFile(zips[0]) as f:
            member = next((m for m in f.namelist() if m.endswith(src_name)), None)
            if member:
                with f.open(member) as s, open(CK / dst_name, 'wb') as d:
                    shutil.copyfileobj(s, d)
INIT = CK / 'floorplan_unetpp_run2.pt'
print('Run 2 checkpoints found:', sorted(p.name for p in CK.glob('*_run2.pt')) or 'NONE (floor plans train from ImageNet)')
os.chdir(W)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips', 'torchmetrics[image]', 'torch-fidelity',
                'diffusers>=0.30', 'transformers', 'accelerate'])

import torch
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
print('facades:', HAS_FACADES, '| cubicasa:', HAS_FP)
assert N_GPU > 0, 'No GPU: Session options > Accelerator > GPU T4 x2'

In [ ]:
# 2) Train: two GPU lanes in parallel (blocks until done; progress every 2 minutes)
PY = sys.executable
P2P = f'{PY} -m src.training.train_facade --mode pix2pix --recipe v3 --epochs 100 --workers 2 --time-budget-min 330'
PARSER = f'{PY} -m src.training.train_facade_parser --epochs 40 --workers 2 --time-budget-min 40'
if INIT.exists():
    FP = (f'{PY} -m src.training.train_floorplan --recipe v3 --init-checkpoint {INIT} --image-size 512 --amp '
          f'--epochs 10 --workers 3 --time-budget-min 240')
else:
    FP = f'{PY} -m src.training.train_floorplan --recipe v3 --image-size 512 --amp --epochs 25 --workers 3 --time-budget-min 300'
CNET = f'{PY} -m src.training.train_controlnet --max-steps 8000 --workers 2 --time-budget-min 270'

lanes = {}
if HAS_FACADES:
    lanes['facades'] = (0, f'{P2P} ; {PARSER}')
if HAS_FP or HAS_FACADES:
    lanes['floorplan_controlnet'] = (1 if N_GPU >= 2 else 0, ' ; '.join(c for c in (FP if HAS_FP else '', CNET if HAS_FACADES else '') if c))
if N_GPU < 2 and len(lanes) == 2:   # single GPU: everything in sequence with smaller caps
    lanes = {'all': (0, f"{P2P.replace('330', '200')} ; {PARSER} ; {FP.replace('240', '150')} ; {CNET.replace('270', '150')}")}

procs = {}
for name, (gpu, cmd) in lanes.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               MLFLOW_TRACKING_URI=f'sqlite:///{W}/mlflow_{name}.db')
    procs[name] = subprocess.Popen(cmd, shell=True, cwd=W, env=env,
                                   stdout=open(W / f'log_{name}.txt', 'w'), stderr=subprocess.STDOUT)
    print(f'started {name} on GPU {gpu}: {cmd}\n')

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    print(f'--- {(time.time() - t0) / 60:.0f} min ---')
    for name in procs:
        lines = (W / f'log_{name}.txt').read_text(errors='ignore').splitlines()
        keep = [l for l in lines if any(k in l for k in ('Epoch', 'budget', 'Error', 'mode=', 'Floorplan]', 'ControlNet]', 'Parser]'))]
        print(f'  {name}: {keep[-1] if keep else (lines[-1] if lines else "starting...")}')
print('exit codes:', {n: p.returncode for n, p in procs.items()}, f'| {(time.time() - t0) / 60:.1f} min')
for name, p in procs.items():
    if p.returncode != 0:
        print(f'\n===== last lines of log_{name}.txt =====')
        print('\n'.join((W / f'log_{name}.txt').read_text(errors='ignore').splitlines()[-30:]))

In [ ]:
# 3) Final evaluation on the TEST split (once). Facade models share one protocol incl. FCN-score.
def run(*args):
    subprocess.run([sys.executable, '-m', *args], cwd=W)

for name in ('facade_pix2pix_best.pt', 'facade_pix2pix_run2.pt', 'facade_unet_l1_run2.pt'):
    if (CK / name).exists():
        print(f'\n===== {name} ====='); run('src.evaluation.evaluate', 'facade', '--checkpoint', str(CK / name))
print('\n===== ControlNet, NOT fine-tuned (what the original project showed) =====')
run('src.evaluation.evaluate_controlnet', '--controlnet', 'lllyasviel/control_v11p_sd15_seg', '--name', 'facade_controlnet_init')
if (W / 'models/controlnet_facade').exists():
    print('\n===== ControlNet, fine-tuned on CMP (Run 3) ====='); run('src.evaluation.evaluate_controlnet')

fp = CK / 'floorplan_unetpp_best.pt'
if fp.exists():
    for label, extra in (('single pass', []), ('4-fold TTA', ['--tta']), ('restyled test', ['--styled'])):
        print(f'\n===== floorplan, {label} ====='); run('src.evaluation.evaluate', 'floorplan', '--checkpoint', str(fp), '--batch-size', '4', *extra)
    print('\n===== floorplan, detection + 3D layout ====='); run('src.evaluation.evaluate_layout', '--checkpoint', str(fp))
    print('\n===== floorplan, external plans ====='); run('src.evaluation.evaluate_external', '--checkpoint', str(fp))
if INIT.exists():   # same restyled test for the Run 2 model, for a fair before/after
    print('\n===== Run 2 floor-plan model, restyled test ====='); run('src.evaluation.evaluate', 'floorplan', '--checkpoint', str(INIT), '--batch-size', '4', '--styled')
print('\n===== GPU latency (T4): plan -> 3D layout, Pix2Pix, ControlNet ====='); run('src.mlops.benchmark_gpu')

In [ ]:
# 4) Scoreboard: Run 3 vs Run 2 vs the original ArchiGen (same CMP / CubiCasa test splits)
M = W / 'models/metrics'
load = lambda n: json.loads((M / n).read_text()) if (M / n).exists() else None
mean = lambda d, k: (d.get(k) or {}).get('mean') if d else None
fcn = lambda d, k='fcn_score': (d.get(k) or {}).get('miou') if d else None
fmt = lambda v, p=4: '   -   ' if v is None else f'{v:.{p}f}'

print('FACADES (test, 106)                 KIDx1000  LPIPS-sq  SSIM    FCN mIoU (layout fidelity)')
print('  original ArchiGen (EMA)            63.55     0.3096    0.2893     -')
rows = [('Run 2 U-Net L1 baseline', 'facade_unet_l1_run2_test.json'), ('Run 2 Pix2Pix', 'facade_pix2pix_run2_test.json'),
        ('Run 3 Pix2Pix (2-scale D)', 'facade_pix2pix_best_test.json'), ('ControlNet NOT fine-tuned', 'facade_controlnet_init_test.json'),
        ('Run 3 ControlNet fine-tuned', 'facade_controlnet_test.json')]
for label, name in rows:
    r = load(name)
    print(f'  {label:<34}', fmt(mean(r, 'kid_x1000'), 2), '   ', fmt(mean(r, 'lpips_squeeze')), fmt(mean(r, 'ssim')), '  ', fmt(fcn(r)))
real = next((load(n) for _, n in rows if load(n)), None)
print(f"  {'real photos (upper bound)':<34}    -         -         -      ", fmt(fcn(real, 'fcn_score_real_photos')))

def fp_row(r):
    if not r: return None
    s = r['structure']['per_class_iou']
    return [100 * r['merged_wall_iou'], 100 * (s['door'] or 0), 100 * (s['window'] or 0), 100 * r['rooms']['miou'], 100 * r['rooms']['pixel_acc']]

print('\nFLOOR PLANS (test, 400)             wall    door    window  room mIoU  room acc')
print('  original ArchiGen                  74.32   62.38   73.28   54.72*     85.85')
print('  Run 2 single pass                  73.35   66.76   77.18   66.64      93.99')
for label, name in (('Run 3 single pass', 'floorplan_unetpp_best_test.json'), ('Run 3 4-fold TTA', 'floorplan_unetpp_best_test_tta.json'),
                    ('Run 3 restyled test', 'floorplan_unetpp_best_test_styled.json'), ('Run 2 restyled test', 'floorplan_unetpp_run2_test_styled.json')):
    row = fp_row(load(name))
    print(f'  {label:<33}', '   '.join('  -  ' if row is None else f'{v:5.2f}' for v in (row or [None] * 5)))
print('* different room class set: indicative only.')

lay = load('floorplan_unetpp_best_test_layout.json')
if lay:
    d, l = lay['detection'], lay['layout']
    print(f"\nDoor F1@24px {d['door']['radius_24px']['f1']:.4f} (Run 2 0.8975, original 0.9045) | window F1@24px {d['window']['radius_24px']['f1']:.4f} (Run 2 0.8629)"
          f" | shells {l['watertight_shell_rate']:.2%} (Run 2 99.25%, original 96.25%) | footprint IoU {l['footprint_iou_mean']:.4f} (Run 2 0.9521)")
    rt = lay.get('room_types', {})
    if rt.get('n_rooms'):
        print(f"Room type correct in the 3D layout: {rt['accuracy']:.2%} of {rt['n_rooms']} rooms (balanced {rt['balanced_accuracy']:.2%})"
              f" | per type: " + ', '.join(f"{k} {v['accuracy']:.0%}" for k, v in rt['per_type'].items()))
ext = load('floorplan_unetpp_best_external.json')
if ext:
    for name, s in ext['sets'].items():
        print(f"external {name:<12} usable {s['usable_rate']:.0%}  empty {s['empty_rate']:.0%}  shell {s['shell_valid_rate']:.0%}")
    print('Run 2: dev20 65% usable | fresh12 83%, 0 empty, shell 100% | modern8 100% | synthetic10 50% | original dev20 55% (85% experimental branch)')
g = load('latency_gpu.json')
if g:
    f = g['floorplan_pipeline']
    print(f"\nGPU latency on {g['device']}: plan -> 3D layout p50 {f['total']['p50_ms']} ms / p95 {f['total']['p95_ms']} ms "
          f"(UNet++ {f['unetpp_inference']['p50_ms']} ms, vectorize {f['vectorize_to_3d_layout_cpu']['p50_ms']} ms) "
          f"| original: 246.5 ms mean / 282.7 ms p95 on RTX 3050")
    print(f"Pix2Pix p50 {g['facade_pix2pix_256']['p50_ms']} ms (original 30.4 ms)"
          + (f" | ControlNet 25 steps p50 {g['facade_controlnet_512_25steps']['p50_ms']} ms" if 'facade_controlnet_512_25steps' in g else ''))

In [ ]:
# 5) Package results -> /kaggle/working/archigen_results_run3.zip (download from the Output tab)
R = Path('/kaggle/working/archigen_results_run3')
if R.exists():
    shutil.rmtree(R)
(R / 'checkpoints').mkdir(parents=True)
for f in CK.glob('*_best.pt'):
    shutil.copy(f, R / 'checkpoints')
if (W / 'models/controlnet_facade').exists():
    shutil.copytree(W / 'models/controlnet_facade', R / 'controlnet_facade')
shutil.copytree(W / 'models/metrics', R / 'metrics')
if (W / 'models/samples').exists():
    shutil.copytree(W / 'models/samples', R / 'samples')
for f in list(W.glob('log_*.txt')) + list(W.glob('mlflow_*.db')):
    shutil.copy(f, R)
shutil.make_archive('/kaggle/working/archigen_results_run3', 'zip', R)
shutil.rmtree(R)
shutil.rmtree(W)
print('Ready: /kaggle/working/archigen_results_run3.zip',
      f"{Path('/kaggle/working/archigen_results_run3.zip').stat().st_size / 1e6:.0f} MB")